# Maia Lite 0.3.0 Small — Architecture & Training Smoke Test

**Purpose:** verify the proposed 20-layer decoder-only architecture, tokenizer compatibility, BF16 forward/backward, GPU memory, throughput, a real-corpus microbatch, and exact checkpoint resume **before freezing the design**.

**Safety:** no full-corpus tokenization, no production training, and no modification of the corpus or freeze. Full-scale test runs only when CUDA is available. The resume regression uses a separate tiny model for speed and strict comparison. Comments and implementation are in English.

**Reference:** `maia_lite_030_small_architecture_and_corpus_policy_v1.md` (design proposal, not yet approved).

## 1. Mount Drive and inspect the runtime

In [ ]:
from pathlib import Path
import os, sys, json, time, math, random, hashlib, platform, gzip, tempfile
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Drive mount skipped.')

SEED = 20261027
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
    print('BF16 supported:', torch.cuda.is_bf16_supported())


## 2. Read the frozen tokenizer (no assumptions about vocabulary size)

In [ ]:
from transformers import AutoTokenizer

RUN = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
FREEZE = RUN / 'freeze_v8_001'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
EXPECTED_FINGERPRINT = '5da873b1198ebe273f4254be9a67a5e12426c796c6026022e8a65c57113a2f07'
OUTPUT = RUN / 'architecture_smoke_test_v1'
OUTPUT.mkdir(parents=True, exist_ok=True)

assert (FREEZE / 'manifest.json').exists(), 'Freeze manifest not found.'
assert TOKENIZER_DIR.exists(), 'Tokenizer directory not found.'
freeze = json.loads((FREEZE / 'manifest.json').read_text(encoding='utf-8'))
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True)
VOCAB_SIZE = len(tok)
EOS_ID = tok.eos_token_id
assert EOS_ID is not None, 'EOS token is required for the packing contract.'
assert EOS_ID == 2, f'Pilot v2 used EOS=2, found {EOS_ID}'
assert VOCAB_SIZE > 1000

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for buf in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(buf)
    return h.hexdigest()

tokenizer_file_hashes = {str(p.relative_to(TOKENIZER_DIR)): sha256_file(p)
                         for p in sorted(TOKENIZER_DIR.rglob('*')) if p.is_file()}
assert tokenizer_file_hashes
# The freeze fingerprint algorithm is not reconstructed here. Record its known
# value and independently verify per-file hashes against the pilot v2 report.
PILOT_V2_REPORT = RUN / 'pretraining_prep_pilot_v2_stratified' / 'pilot_report_v2.json'
if PILOT_V2_REPORT.exists():
    prior = json.loads(PILOT_V2_REPORT.read_text(encoding='utf-8'))
    for name, expected in prior.get('tokenizer_file_sha256', {}).items():
        assert tokenizer_file_hashes.get(name) == expected, f'Tokenizer file changed: {name}'
    print('Tokenizer file hashes match pilot v2: PASS')
else:
    print('Pilot v2 report unavailable: per-file cross-check skipped.')
print('Vocabulary:', VOCAB_SIZE, '| EOS:', EOS_ID, '| PAD:', tok.pad_token_id)
print('Known candidate fingerprint:', EXPECTED_FINGERPRINT)
print('Historical tokenizer identity: NOT PROVEN')


## 3. Define the model precisely

The implementation uses pre-norm RMSNorm, RoPE with base 10,000, causal scaled-dot-product attention, SwiGLU, tied embeddings, and a model-internal one-token label shift. **Input length is 2,048 tokens; no 2,049-token requirement.** No dropout is used in this baseline, simplifying deterministic resume testing.

In [ ]:
@dataclass(frozen=True)
class MaiaConfig:
    vocab_size: int
    n_layers: int = 20
    d_model: int = 1024
    n_heads: int = 16
    n_kv_heads: int = 16
    ffn_dim: int = 2752
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    init_std: float = 0.02

    def validate(self):
        assert self.d_model % self.n_heads == 0
        assert self.n_heads == self.n_kv_heads, 'This baseline implements MHA only.'
        assert (self.d_model // self.n_heads) % 2 == 0, 'RoPE requires even head dimension.'
        assert self.vocab_size > 0 and self.max_seq_len >= 2

class RMSNorm(nn.Module):
    def __init__(self, width, eps):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight

def rope_cos_sin(seq_len, head_dim, theta, device, dtype):
    # Frequencies are generated in FP32 for numerical stability.
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))
    phases = torch.outer(torch.arange(seq_len, device=device, dtype=torch.float32), inv)
    return phases.cos()[None, None, :, :].to(dtype), phases.sin()[None, None, :, :].to(dtype)

def apply_rope(x, cos, sin):
    # Interleaved even/odd RoPE convention, explicitly fixed by this implementation.
    even, odd = x[..., ::2], x[..., 1::2]
    return torch.stack((even*cos - odd*sin, even*sin + odd*cos), dim=-1).flatten(-2)

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.theta = cfg.rope_theta
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
    def forward(self, x):
        b, t, d = x.shape
        def heads(proj): return proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        q, k, v = heads(self.q_proj), heads(self.k_proj), heads(self.v_proj)
        cos, sin = rope_cos_sin(t, self.head_dim, self.theta, x.device, q.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        z = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        return self.o_proj(z.transpose(1, 2).contiguous().view(b, t, d))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.down = nn.Linear(cfg.ffn_dim, cfg.d_model, bias=False)
    def forward(self, x): return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)
    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))

class MaiaLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # true parameter tying
        self.apply(self._init)
    def _init(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)
    def forward(self, ids, labels=None):
        assert ids.ndim == 2 and 2 <= ids.shape[1] <= self.cfg.max_seq_len
        x = self.embed(ids)
        for layer in self.blocks: x = layer(x)
        logits = self.lm_head(self.norm(x))
        if labels is None: return logits, None
        assert labels.shape == ids.shape
        # FP32 cross-entropy, shifted internally: t predicts t+1.
        loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, self.cfg.vocab_size),
                               labels[:, 1:].reshape(-1), ignore_index=-100)
        return logits, loss

CFG = MaiaConfig(vocab_size=VOCAB_SIZE)
CFG.validate()
print('Proposed configuration:', json.dumps(asdict(CFG), indent=2))


## 4. Exact parameter count and architectural invariants

In [ ]:
# Instantiate on the meta device to count parameters without allocating GPU RAM.
with torch.device('meta'):
    model_meta = MaiaLM(CFG)
param_count = sum(p.numel() for p in model_meta.parameters())
embedding_count = CFG.vocab_size * CFG.d_model
assert model_meta.embed.weight is model_meta.lm_head.weight
assert len(model_meta.blocks) == 20
assert CFG.d_model // CFG.n_heads == 64
print(f'Exact trainable parameters: {param_count:,} ({param_count / 1e6:.3f} M)')
print(f'Tied embedding parameters: {embedding_count:,}')
print(f'FP32 weights alone: {param_count * 4 / 2**30:.2f} GiB')
print(f'BF16 weights alone: {param_count * 2 / 2**30:.2f} GiB')


## 5. Full model on GPU — synthetic forward/backward and throughput

Requires CUDA. Uses batch size 1 and the full 2,048-token context. This is a **smoke test**, not a throughput benchmark with stable statistical confidence. Full model is deleted after the test to avoid VRAM pressure. If out of memory, the exception is recorded and the rest of the notebook continues.

In [ ]:
full_result = {'status': 'NOT_RUN'}
if DEVICE.type != 'cuda':
    full_result = {'status': 'SKIPPED_NO_CUDA'}
    print('No CUDA GPU; full-model training step skipped. Tiny resume test will run on CPU.')
else:
    model = None
    optimizer = None
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        torch.manual_seed(SEED)
        model = MaiaLM(CFG).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
        use_bf16 = torch.cuda.is_bf16_supported()
        dtype = torch.bfloat16 if use_bf16 else torch.float16
        if not use_bf16:
            print('WARNING: BF16 unavailable. FP16 smoke test requires GradScaler.')
        scaler = torch.amp.GradScaler('cuda', enabled=not use_bf16)
        ids = torch.randint(0, VOCAB_SIZE, (1, CFG.max_seq_len), device=DEVICE)
        ids[:, -1] = EOS_ID
        model.train(); optimizer.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.autocast('cuda', dtype=dtype):
            _, loss = model(ids, ids)
        assert torch.isfinite(loss), f'Non-finite loss: {loss.item()}'
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        assert torch.isfinite(grad_norm), 'Non-finite gradient norm'
        scaler.step(optimizer); scaler.update()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        full_result = {
            'status': 'PASS', 'loss': float(loss.item()),
            'gradient_norm_before_clip': float(grad_norm.item()),
            'seconds_per_step': elapsed,
            'effective_predicted_tokens': CFG.max_seq_len - 1,
            'predicted_tokens_per_second': (CFG.max_seq_len - 1) / elapsed,
            'peak_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
            'peak_reserved_gib': torch.cuda.max_memory_reserved() / 2**30,
            'dtype': str(dtype), 'batch_size': 1,
        }
        print(json.dumps(full_result, indent=2))
    except torch.cuda.OutOfMemoryError as exc:
        full_result = {'status': 'OOM', 'error': str(exc)[:500]}
        print('OOM during full smoke test:', full_result['error'])
    finally:
        del model, optimizer
        import gc
        gc.collect()
        torch.cuda.empty_cache()


## 6. Optional real-corpus microbatch

To avoid scanning 21,021 compressed files, specify the path to **one** existing `.jsonl.gz` chunk in `REAL_CHUNK_PATH`. This reads only the first few valid documents and does not write into the source corpus. Uses a smaller model to validate the full text → tokenizer → EOS → loss path; it does not benchmark the full model on real data.

In [ ]:
REAL_CHUNK_PATH = ''  # e.g. '/content/drive/MyDrive/.../fineweb2_pt_chunk_000001.jsonl.gz'
real_result = {'status': 'SKIPPED_NO_PATH'}
if REAL_CHUNK_PATH:
    p = Path(REAL_CHUNK_PATH)
    assert p.is_file() and p.name.endswith('.jsonl.gz'), 'Set a valid gzip JSONL chunk path.'
    tokens = []
    docs = 0
    with gzip.open(p, 'rt', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            text = obj.get('text')
            if not isinstance(text, str) or not text.strip(): continue
            tokens.extend(tok.encode(text, add_special_tokens=False))
            tokens.append(EOS_ID)
            docs += 1
            if len(tokens) >= 257 or docs >= 100: break
    assert len(tokens) >= 2, 'Not enough real tokens for causal loss.'
    # Small network for end-to-end functional validation.
    real_cfg = MaiaConfig(VOCAB_SIZE, n_layers=2, d_model=128, n_heads=4,
                          n_kv_heads=4, ffn_dim=352, max_seq_len=256)
    real_model = MaiaLM(real_cfg).to(DEVICE)
    real_ids = torch.tensor(tokens[:256], device=DEVICE, dtype=torch.long).unsqueeze(0)
    real_optimizer = torch.optim.AdamW(real_model.parameters(), lr=1e-4)
    _, real_loss = real_model(real_ids, real_ids)
    assert torch.isfinite(real_loss)
    real_loss.backward(); real_optimizer.step()
    real_result = {'status':'PASS', 'documents_read':docs,
                   'tokens_read':len(tokens), 'loss':float(real_loss.item()),
                   'source_chunk':p.name}
    print(json.dumps(real_result, indent=2))
    del real_model, real_optimizer
else:
    print('Real-corpus test skipped. Set REAL_CHUNK_PATH to enable.')


## 7. Exact checkpoint-resume regression

This test compares **three uninterrupted optimizer steps** against **one step + save + reload + two steps** on a small CPU model. It verifies model weights, AdamW state, scheduler, Python/NumPy/Torch RNG, and the next-batch cursor. The CPU run is intentional: it avoids nondeterministic CUDA kernels and makes an exact equality assertion meaningful. CUDA resume equivalence requires an additional GPU-specific test before production.

In [ ]:
import copy

resume_cfg = MaiaConfig(vocab_size=257, n_layers=2, d_model=64, n_heads=4,
                        n_kv_heads=4, ffn_dim=176, max_seq_len=32)
resume_device = torch.device('cpu')

# Fixed data stream with a cursor that is explicitly persisted.
data_gen = torch.Generator(device='cpu').manual_seed(12345)
batches = [torch.randint(0, resume_cfg.vocab_size, (2, 32), generator=data_gen)
           for _ in range(4)]

def setup_run():
    random.seed(777); np.random.seed(777); torch.manual_seed(777)
    m = MaiaLM(resume_cfg).to(resume_device)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda step: 1.0 - 0.05 * step)
    return m, opt, sch

def run_step(m, opt, sch, cursor):
    m.train(); opt.zero_grad(set_to_none=True)
    _, loss = m(batches[cursor], batches[cursor])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step(); sch.step()
    return float(loss.item()), cursor + 1

def capture_rng():
    return {'python':random.getstate(), 'numpy':np.random.get_state(),
            'torch':torch.get_rng_state()}

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy'])
    torch.set_rng_state(s['torch'])

m_a, o_a, s_a = setup_run()
cursor_a = 0; losses_a = []
for _ in range(3):
    loss, cursor_a = run_step(m_a, o_a, s_a, cursor_a)
    losses_a.append(loss)

m_b, o_b, s_b = setup_run()
cursor_b = 0
loss, cursor_b = run_step(m_b, o_b, s_b, cursor_b)
losses_b = [loss]
checkpoint = {
    'model': m_b.state_dict(), 'optimizer': o_b.state_dict(),
    'scheduler': s_b.state_dict(), 'rng': capture_rng(),
    'cursor': cursor_b, 'global_step': 1,
    'config': asdict(resume_cfg),
}
checkpoint_path = Path(tempfile.gettempdir()) / 'maia_lite_resume_smoke_v1.pt'
torch.save(checkpoint, checkpoint_path)
del m_b, o_b, s_b, checkpoint

m_c, o_c, s_c = setup_run()
# weights_only=False is safe ONLY because this file was created locally by this cell.
state = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
assert state['config'] == asdict(resume_cfg)
m_c.load_state_dict(state['model']); o_c.load_state_dict(state['optimizer'])
s_c.load_state_dict(state['scheduler']); restore_rng(state['rng'])
cursor_c = state['cursor']
assert cursor_c == state['global_step'] == 1
for _ in range(2):
    loss, cursor_c = run_step(m_c, o_c, s_c, cursor_c)
    losses_b.append(loss)

assert cursor_a == cursor_c == 3
assert losses_a == losses_b, f'Loss divergence: {losses_a} vs {losses_b}'
assert all(torch.equal(m_a.state_dict()[k], m_c.state_dict()[k])
           for k in m_a.state_dict()), 'Model state differs after resume'
assert o_a.param_groups[0]['lr'] == o_c.param_groups[0]['lr']
resume_result = {'status':'PASS', 'losses_uninterrupted':losses_a,
                 'losses_resumed':losses_b, 'cursor':cursor_c,
                 'weights_bitwise_equal':True, 'optimizer_lr_equal':True,
                 'scope':'tiny CPU deterministic test'}
print(json.dumps(resume_result, indent=2))


## 8. Write a reproducible report and review the approval gates

A passing CPU resume test **does not** prove exact GPU resume. A passing synthetic step **does not** prove convergence. No architectural freeze is performed automatically.

In [ ]:
report = {
    'schema_version': '1.0', 'purpose': 'architecture smoke test, not production training',
    'created_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
    'architecture_proposal': asdict(CFG),
    'parameter_count_exact': param_count,
    'tokenizer_candidate_fingerprint': EXPECTED_FINGERPRINT,
    'tokenizer_file_sha256': tokenizer_file_hashes,
    'historical_tokenizer_identity_verified': False,
    'freeze_id': 'maia_corpus_v8_001',
    'full_model_synthetic': full_result,
    'real_corpus_microbatch': real_result,
    'checkpoint_resume_regression': resume_result,
    'environment': {'python':platform.python_version(), 'torch':torch.__version__,
                    'cuda':torch.version.cuda, 'device':str(DEVICE),
                    'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None},
    'limitations': [
        'Synthetic performance is a one-step measurement, not steady-state throughput.',
        'Checkpoint resume equality was tested on a tiny CPU model, not full GPU training.',
        'Historical tokenizer identity remains unproven.',
        'Real corpus test is optional and requires an explicit chunk path.',
        'Dataset licensing, near-duplicate leakage and quality are outside this test.',
        'Architecture and 2048-token context remain proposals pending review.',
    ],
}
report_path = OUTPUT / 'architecture_smoke_report_v1.json'
if report_path.exists():
    raise FileExistsError(f'Report exists; preserve it and choose a new output path: {report_path}')
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Report saved:', report_path)
print('SHA-256:', sha256_file(report_path))
print('FULL GPU:', full_result['status'])
print('REAL CORPUS:', real_result['status'])
print('RESUME:', resume_result['status'])
print('Next gate: review memory and speed, then GPU resume + representative real-data pilot.')


## Interpretation guide

- **PASS / full model:** the proposed architecture can execute one forward/backward optimizer step at 2,048 tokens on this GPU. It is not yet approved for full training.
- **OOM:** do not silently shrink the architecture; record the failure and decide on activation checkpointing, smaller microbatch, or a revised design.
- **Resume PASS:** exact CPU-state restoration is verified; GPU checkpoint restoration remains a separate gate.
- **Real-data SKIPPED:** provide a specific chunk path and rerun the optional cell to verify actual corpus text and EOS behavior.
- **Production remains blocked** until the dataset split, near-duplicate checks, license review, and resumable preprocessing have been addressed.
